# Why is the error for unseen ion combinations lower than for new compositions? (Part 2, Sect. 3.1)

Table 1 reports a pooled MAE of 0.058 for the three held-out ion combinations, lower than the 0.069 for new
compositions of seen combinations. This notebook loads the five saved networks (`models/`) and the training data,
rebuilds the splits of `01_part2_revision.ipynb` (section I.1), and compares the composition of the two test
sets and their errors at equal ionic strength. Run time: a few seconds.

In [1]:

import numpy as np, pandas as pd, torch, torch.nn as nn
df = pd.read_parquet("training_data.parquet")
ck = torch.load("models/inorganic_seed0.pt", weights_only=False)
IN, OUT = ck["in_cols"], ck["out_cols"]; ions = [c[2:] for c in IN[:-1]]
df = df[df[OUT[1:]].notna().any(axis=1)].reset_index(drop=True)
HELD = [frozenset({"Ca2+", "CO3--"}), frozenset({"NH4+", "HSO4-"}), frozenset({"K+", "IO3-"})]
combo = df.apply(lambda r: frozenset(n for n in ions if r[f"m_{n}"] > 0), axis=1)
gen = combo.isin(HELD).values; rest = np.where(~gen)[0]; p = np.random.RandomState(0).permutation(len(rest))
interp = rest[p[:int(0.1 * len(rest))]]; train = rest[p[int(0.1 * len(rest)):]]; gi = np.where(gen)[0]
X = np.concatenate([np.log1p(df[IN[:-1]].to_numpy(np.float32)), df[["T_K"]].to_numpy(np.float32)], 1)
Y = df[OUT].to_numpy(np.float32); M = np.isfinite(Y); Yz = np.nan_to_num(Y)
I = df.ionic_strength.values; nion = combo.apply(len).values; axis = df.axis.values
print(f"train {len(train)}, interpolation {len(interp)}, unseen combinations {len(gi)}")
for name, idx in [("interpolation", interp), ("unseen combinations", gi)]:
    print(f"{name:20s}: >2 ions {100*np.mean(nion[idx] > 2):4.0f} %, CaSO4(s) axis {100*np.mean(axis[idx] == 'caso4_precip'):4.0f} %, "
          f"I <= 0.1 {100*np.mean(I[idx] <= 0.1):4.0f} %, 0.1-1 {100*np.mean((I[idx] > 0.1) & (I[idx] <= 1)):4.0f} %, "
          f"> 1 {100*np.mean(I[idx] > 1):4.0f} %, median I {np.median(I[idx]):.2f} mol/kg")


train 10420, interpolation 1157, unseen combinations 423
interpolation       : >2 ions   41 %, CaSO4(s) axis   26 %, I <= 0.1   37 %, 0.1-1   35 %, > 1   28 %, median I 0.25 mol/kg
unseen combinations : >2 ions    0 %, CaSO4(s) axis    0 %, I <= 0.1   62 %, 0.1-1   25 %, > 1   13 %, median I 0.03 mol/kg


In [2]:

class MLP(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(15, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 15))
    def forward(self, x): return self.net(x)
class Surrogate(nn.Module):
    def __init__(self):
        super().__init__(); self.net = MLP(); self.register_buffer("ym", torch.zeros(15)); self.register_buffer("ys", torch.ones(15))
    def forward(self, x): return self.net(x) * self.ys + self.ym
def pooled(P, idx, sel):
    E = np.abs(P[idx] - Yz[idx]); m = M[idx] & sel[idx][:, None]; return E[m].mean()
BINS = [(0, 0.1), (0.1, 1), (1, 100)]
rows = []
for s in range(5):
    ck = torch.load(f"models/inorganic_seed{s}.pt", weights_only=False)
    net = Surrogate(); net.load_state_dict(ck["state_dict"]); net.eval()
    with torch.no_grad(): P = net(torch.tensor((X - ck["x_mean"]) / ck["x_std"], dtype=torch.float32)).numpy()
    allr = np.ones(len(df), bool)
    r = {"seed": s, "interpolation, all": pooled(P, interp, allr), "unseen, all": pooled(P, gi, allr)}
    for lo, hi in BINS:
        b = (I > lo) & (I <= hi)
        r[f"interpolation, I {lo}-{hi}"] = pooled(P, interp, b); r[f"unseen, I {lo}-{hi}"] = pooled(P, gi, b)
    rows.append(r)
res = pd.DataFrame(rows).drop(columns="seed")
display(res.agg(["mean", "std"]).T.round(3).rename(columns={"mean": "pooled MAE (mean of 5 seeds)", "std": "sd"}))


,pooled MAE (mean of 5 seeds),sd
"interpolation, all",0.069,0.008
"unseen, all",0.058,0.006
"interpolation, I 0-0.1",0.037,0.007
"unseen, I 0-0.1",0.046,0.006
"interpolation, I 0.1-1",0.043,0.007
"unseen, I 0.1-1",0.045,0.007
"interpolation, I 1-100",0.131,0.023
"unseen, I 1-100",0.138,0.012


The held-out set contains only single-salt solutions, mostly dilute. At equal ionic strength its errors are
equal to or slightly higher than those for new compositions of seen combinations, so the lower pooled value
reflects an easier test set, not better generalization.